# run_accuracy.ipynb

Evaluate **base model accuracy** on DATA JSONL using the **all_at_once** prompt.

What this notebook does:
1. Auto-detect repo root.
2. Load DATA JSONL (no prompts inside).
3. Load `system.txt` and `user_template.txt` from `02_prompts/all_at_once/`.
4. Build messages for each example.
5. Call a model (OpenAI by default; pluggable).
6. Parse strict JSON outputs.
7. Compute accuracy + precision/recall/F1 per label.
8. Save a run folder under:
   `datasets/<dataset_id>/03_experiments/all_at_once/<provider>/<model>/runs/<timestamp>/`

You must set your API key in the environment (recommended) or in the notebook.


In [1]:
# --- 0) Imports ---
import os, json, time, re
from pathlib import Path
from datetime import datetime
import pandas as pd


## 1) Repo root + dataset selection

In [2]:
def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start).resolve()
    for p in [start] + list(start.parents):
        if (p / ".git").exists() or (p / "README.md").exists():
            return p
    raise RuntimeError(f"Repo root not found from start={start}")

repo_root = find_repo_root()
print("Repo root:", repo_root)

# --- EDIT THIS ---
DATASET_ID = "yt_factlink"  # folder name under datasets/

dataset_root = repo_root / "datasets" / DATASET_ID
assert dataset_root.exists(), f"Dataset root not found: {dataset_root}"
print("Dataset root:", dataset_root)


Repo root: /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/ici01_PublicOpinionAnalyzer
Dataset root: /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/ici01_PublicOpinionAnalyzer/datasets/yt_factlink


## 2) Paths (DATA JSONL + prompts)

In [5]:
# --- DATA JSONL (canonical) ---
DATA_JSONL = dataset_root / "01_conversion" / "outputs" / "manual_labels_386_v2.data.jsonl"
assert DATA_JSONL.exists(), f"DATA JSONL missing: {DATA_JSONL}"

# --- PROMPTS ---
PROMPT_DIR = dataset_root / "02_prompts" / "all_at_once"
SYSTEM_TXT = PROMPT_DIR / "system.txt"
USER_TEMPLATE_TXT = PROMPT_DIR / "user.txt"

assert SYSTEM_TXT.exists(), f"system.txt missing: {SYSTEM_TXT}"
assert USER_TEMPLATE_TXT.exists(), f"user_template.txt missing: {USER_TEMPLATE_TXT}"

system_prompt = SYSTEM_TXT.read_text(encoding="utf-8")
user_template = USER_TEMPLATE_TXT.read_text(encoding="utf-8")

print("Loaded system prompt chars:", len(system_prompt))
print("Loaded user template chars:", len(user_template))


Loaded system prompt chars: 2934
Loaded user template chars: 90


## 3) Load DATA JSONL

In [6]:
def read_jsonl(path: Path):
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

data_rows = read_jsonl(DATA_JSONL)
print("Examples:", len(data_rows))
data_rows[0]


Examples: 386


{'VIDEOTITEL': 'The Chinese representative angrily said "Taiwan belongs to China"!',
 'VIDEOCONTEXT': "At the 78th Special Session of the UN General Assembly, China's representative forcefully asserted the one-China principle, claiming Taiwan as inalienable sovereign territory and warning against independence. Japan's representative, Ono Kosei, countered calmly by asking a profound question: if Taiwan truly belongs to China, why is military force necessary for unification? He urged the assembly to respect the principle of self-determination and the wishes of the 23 million Taiwanese people. This thoughtful challenge disrupted the aggressive momentum, sparking a moment of silence and later earning support from U.S. and European delegates. Following the public impasse, the Chinese and Japanese diplomats shared a rare private moment, candidly discussing their difficult positions and expressing a mutual hope for future peace. The entire incident rekindled global debate on sovereignty, demo

## 4) Build user message from placeholders

In [17]:
def build_user_message(row: dict) -> str:
    return (user_template
            .replace("[VIDEOTITLE]", row.get("VIDEOTITEL",""))
            .replace("[VIDEOCONTEXT]", row.get("VIDEOCONTEXT",""))
            .replace("[TARGETCOMMENT]", row.get("TARGETCOMMENT","")))

# preview one built message
print(build_user_message(data_rows[0])[:4000])


Input
Title:
The Chinese representative angrily said "Taiwan belongs to China"!

Video Context:
At the 78th Special Session of the UN General Assembly, China's representative forcefully asserted the one-China principle, claiming Taiwan as inalienable sovereign territory and warning against independence. Japan's representative, Ono Kosei, countered calmly by asking a profound question: if Taiwan truly belongs to China, why is military force necessary for unification? He urged the assembly to respect the principle of self-determination and the wishes of the 23 million Taiwanese people. This thoughtful challenge disrupted the aggressive momentum, sparking a moment of silence and later earning support from U.S. and European delegates. Following the public impasse, the Chinese and Japanese diplomats shared a rare private moment, candidly discussing their difficult positions and expressing a mutual hope for future peace. The entire incident rekindled global debate on sovereignty, democracy, 

## 5) Model client (OpenAI default)

If you're not using OpenAI, replace this section with your provider's SDK.


In [29]:
import json

# --- Load API Keys ---
KEYS_PATH = repo_root / "keys.json"
if not KEYS_PATH.exists():
    raise FileNotFoundError(f"keys.json missing: {KEYS_PATH}")

keys = json.loads(KEYS_PATH.read_text())

OPENAI_API_KEY = keys.get("openai")
if not OPENAI_API_KEY:
    raise ValueError("Missing 'openai' key in keys.json")

# --- OpenAI Client Setup ---
%pip install openai
from openai import OpenAI

client = OpenAI(api_key=OPENAI_API_KEY)

PROVIDER = "openai"
MODEL_NAME = "gpt-4.1-2025-04-14"  # <-- change as needed

#-- Define Structured Output Model ---
from typing import List
from pydantic import BaseModel

class LabelOutput(BaseModel):
    C1: int
    C2: int
    C3: int
    C4: int
    C6: int

#--- Function to Call Model ---
def call_model(messages, temperature=0.0):
    resp = client.responses.parse(
        model=MODEL_NAME,
        input=messages,          # same messages list as before
        text_format=LabelOutput, # schema enforcement + parsing
        temperature=temperature,
    )
    return resp.output_parsed   # <-- already a LabelOutput instance



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 6) Run inference

In [33]:
from IPython.display import clear_output


LABEL_KEYS = ["C1","C2","C3","C4","C6"]  # adjust if you add more

predictions = []
failures = 0

predictions = []
failures = 0

for i, row in enumerate(data_rows):
    user_msg = build_user_message(row)
    msgs = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_msg},
    ]

    try:
        out = call_model(msgs)        # returns LabelOutput instance
    except Exception as e:
        # API or schema failure
        failures += 1
        out = None

    if out is None:
        # Treat as zero predictions
        pred = {k: 0 for k in LABEL_KEYS}
        raw_output = "<PARSE FAILURE>"
    else:
        # Build prediction dict from Pydantic object
        pred = {k: int(getattr(out, k)) for k in LABEL_KEYS}
        raw_output = out.model_dump_json()

    predictions.append({
        "pred": pred,
        "gold": row["labels"],
        "raw_output": raw_output,
    })

    clear_output(wait=True)
    print(f"[{i+1}/{len(data_rows)}]")
    print(row.get("TARGETCOMMENT"))
    print("GOLD:", row["labels"])
    print("PRED:", pred)
    
    # if (i+1) % 25 == 0:
    #     print(f"Processed {i+1}/{len(data_rows)} ... failures so far: {failures}")
    #     time.sleep(0.3)

print("Done. Total failures:", failures)
predictions[0]



[386/386]
可惜一堆中國人腦筋不清楚，還支持中共的髒手來污染這片中華文化淨土😢
GOLD: {'C1': 1, 'C2': 1, 'C3': 0, 'C4': 0, 'C6': 0}
PRED: {'C1': 1, 'C2': 1, 'C3': 0, 'C4': 0, 'C6': 0}
Done. Total failures: 0


{'pred': {'C1': 1, 'C2': 1, 'C3': 0, 'C4': 0, 'C6': 0},
 'gold': {'C1': 0, 'C2': 0, 'C3': 0, 'C4': 0, 'C6': 0},
 'raw_output': '{"C1":1,"C2":1,"C3":0,"C4":0,"C6":0}'}

## 7) Compute metrics

In [34]:
def compute_metrics(predictions):
    # per-label confusion counts
    counts = {k: {"tp":0,"fp":0,"tn":0,"fn":0} for k in LABEL_KEYS}

    for ex in predictions:
        gold = ex["gold"]
        pred = ex["pred"]
        for k in LABEL_KEYS:
            g = int(gold.get(k,0))
            p = int(pred.get(k,0))
            if p==1 and g==1: counts[k]["tp"] += 1
            elif p==1 and g==0: counts[k]["fp"] += 1
            elif p==0 and g==0: counts[k]["tn"] += 1
            elif p==0 and g==1: counts[k]["fn"] += 1

    metrics = {}
    for k,c in counts.items():
        tp,fp,tn,fn = c["tp"],c["fp"],c["tn"],c["fn"]
        prec = tp/(tp+fp) if (tp+fp)>0 else 0.0
        rec  = tp/(tp+fn) if (tp+fn)>0 else 0.0
        f1   = (2*prec*rec/(prec+rec)) if (prec+rec)>0 else 0.0
        acc  = (tp+tn)/(tp+tn+fp+fn) if (tp+tn+fp+fn)>0 else 0.0
        metrics[k] = {"precision":prec, "recall":rec, "f1":f1, "accuracy":acc, **c}

    # macro averages
    macro_f1 = sum(metrics[k]["f1"] for k in LABEL_KEYS)/len(LABEL_KEYS)
    macro_acc = sum(metrics[k]["accuracy"] for k in LABEL_KEYS)/len(LABEL_KEYS)

    return metrics, {"macro_f1": macro_f1, "macro_accuracy": macro_acc}

per_label, overall = compute_metrics(predictions)
per_label, overall


({'C1': {'precision': 0.8,
   'recall': 0.8590604026845637,
   'f1': 0.8284789644012946,
   'accuracy': 0.8626943005181347,
   'tp': 128,
   'fp': 32,
   'tn': 205,
   'fn': 21},
  'C2': {'precision': 0.8279569892473119,
   'recall': 0.8369565217391305,
   'f1': 0.8324324324324325,
   'accuracy': 0.9196891191709845,
   'tp': 77,
   'fp': 16,
   'tn': 278,
   'fn': 15},
  'C3': {'precision': 0.4583333333333333,
   'recall': 0.5789473684210527,
   'f1': 0.5116279069767442,
   'accuracy': 0.9455958549222798,
   'tp': 11,
   'fp': 13,
   'tn': 354,
   'fn': 8},
  'C4': {'precision': 0.7121212121212122,
   'recall': 0.9215686274509803,
   'f1': 0.8034188034188035,
   'accuracy': 0.9404145077720207,
   'tp': 47,
   'fp': 19,
   'tn': 316,
   'fn': 4},
  'C6': {'precision': 0.0,
   'recall': 0.0,
   'f1': 0.0,
   'accuracy': 0.9948186528497409,
   'tp': 0,
   'fp': 2,
   'tn': 384,
   'fn': 0}},
 {'macro_f1': 0.5951916214458549, 'macro_accuracy': 0.9326424870466321})

## 8) Save run folder

In [35]:
timestamp = datetime.now().strftime("%Y-%m-%d_%H%M%S")

run_dir = (dataset_root / "03_experiments" / "all_at_once" /
           PROVIDER / MODEL_NAME / "runs" / timestamp)
run_dir.mkdir(parents=True, exist_ok=True)

# Build compact per-class summary for config
per_label_summary = {
    k: {
        "accuracy": per_label[k]["accuracy"],
        "f1": per_label[k]["f1"],
        "precision": per_label[k]["precision"],
        "recall": per_label[k]["recall"],
        "tp": per_label[k]["tp"],
        "fp": per_label[k]["fp"],
        "tn": per_label[k]["tn"],
        "fn": per_label[k]["fn"],
    }
    for k in LABEL_KEYS
}

# save config
run_config = {
    "dataset_id": DATASET_ID,
    "experiment_type": "all_at_once",
    "provider": PROVIDER,
    "model_name": MODEL_NAME,
    "data_jsonl": str(DATA_JSONL.relative_to(repo_root)),
    "prompt_dir": str(PROMPT_DIR.relative_to(repo_root)),
    "timestamp": timestamp,
    "failures": failures,
    "overall": overall,
    "per_label": per_label_summary,

}
(run_dir / "run_config.json").write_text(json.dumps(run_config, indent=2, ensure_ascii=False), encoding="utf-8")

# save preds
with (run_dir / "preds.jsonl").open("w", encoding="utf-8") as f:
    for ex in predictions:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

# save metrics
(run_dir / "metrics.json").write_text(json.dumps(per_label, indent=2, ensure_ascii=False), encoding="utf-8")

# --- save prompt snapshot ---
prompt_snapshot_dir = run_dir / "prompt_snapshot"
prompt_snapshot_dir.mkdir(exist_ok=True)

# copy system + user prompts EXACTLY as used
(prompt_snapshot_dir / "system.txt").write_text(system_prompt, encoding="utf-8")
(prompt_snapshot_dir / "user_template.txt").write_text(user_template, encoding="utf-8")

print("Saved run to:", run_dir)


Saved run to: /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/ici01_PublicOpinionAnalyzer/datasets/yt_factlink/03_experiments/all_at_once/openai/gpt-4.1-2025-04-14/runs/2025-11-24_014531


## Next steps

- To run **single_class** experiments, copy this notebook and switch:
  - `PROMPT_DIR` → `02_prompts/single_class`
  - save under `03_experiments/single_class/...`
- For finetuned models, change `MODEL_NAME` to your finetuned id and re-run.
